# Dengue, saneamento e clima — agregação por UF e mês

Este notebook reúne **SINAN/DATASUS**, **IBGE/SIDRA** e **INMET** em uma grade
de **27 UFs × 12 meses**. O recorte inicial é **2010**; a configuração também
aceita **2022**, analisado separadamente porque as definições censitárias diferem.

Produtos: base mensal, resumo anual por UF, auditorias, metadados e gráficos interativos.
O desfecho principal é a **taxa de notificações confirmadas por 100 mil residentes**;
notificações totais permanecem disponíveis para análise complementar.

**Execução:** selecione o ambiente Python do projeto e execute as células em ordem.
Na primeira execução, os três indicadores estaduais de saneamento são consultados no
SIDRA e armazenados em cache. Se o arquivo INMET estiver ausente, o notebook executa
a integração SINAN–IBGE, mantém o clima nulo e identifica essa ausência nos resultados.

## 1. Configuração

- `ANO`: 2010 ou 2022, com população e arquivos SINAN disponíveis.
- `UF_DESTAQUE`: estado mostrado no painel temporal.
- `MES_DISPERSAO`: mês da comparação entre UFs com as três fontes (1 a 12; padrão: março).
- `CAMINHO_INMET`: Parquet produzido pelo ingestor do projeto.
- `ESTACOES_FIXAS`: número de estações do painel por UF, selecionadas somente pela cobertura.
- Temperatura e umidade exigem pelo menos 80% dos dias do mês; chuva acumulada exige 100%.

A preparação das fontes é feita na raiz do projeto:

```bash
uv run python -m utils.analysis_pipeline --refresh-ibge --allow-undated
uv run python -m utils.inmet_pipeline data/inmet_bruto --csv-output data/INMET_dataset_agregado.csv
```

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "utils/analysis_pipeline.py").exists()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import polars as pl
import numpy as np
from base64 import b64decode
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown
from utils.aggregate_sources import (
    SANITATION, WEATHER, build_aggregations, export_aggregations,
)

ANO = 2010
UF_DESTAQUE = "SP"
MES_DISPERSAO = 3
CAMINHO_INMET = PROJECT_ROOT / "data/INMET_dataset_agregado.parquet"
ATUALIZAR_SIDRA = False
ESTACOES_FIXAS = 3
COBERTURA_MINIMA_ESTACAO = 0.80
COBERTURA_MINIMA_MENSAL = 0.80
DESTINO = PROJECT_ROOT / "sinan/processados/agregacao_tres_fontes" / str(ANO)

MESES = ["Jan", "Fev", "Mar", "Abr", "Mai", "Jun", "Jul", "Ago", "Set", "Out", "Nov", "Dez"]
ROTULOS_SANEAMENTO = {
    "pct_agua_rede": "Água — rede geral" if ANO == 2010 else "Água — rede geral como fonte principal",
    "pct_esgotamento": "Esgoto — rede ou fossa séptica" if ANO == 2010 else "Esgoto — rede/pluvial ou fossa ligada à rede",
    "pct_coleta_lixo": "Lixo — coleta direta ou em caçamba",
}
CORES_REGIAO = {"Norte": "#228b69", "Nordeste": "#d19524", "Centro-Oeste": "#a86635", "Sudeste": "#2878b5", "Sul": "#8055a5"}
px.defaults.template = "plotly_white"
FIGURAS = {}

def dados_portaveis(valor):
    if isinstance(valor, dict):
        if "bdata" in valor and "dtype" in valor:
            matriz = np.frombuffer(b64decode(valor["bdata"]), dtype=valor["dtype"])
            if "shape" in valor:
                matriz = matriz.reshape(tuple(int(n) for n in valor["shape"].split(",")))
            return matriz.tolist()
        return {chave: dados_portaveis(item) for chave, item in valor.items()}
    if isinstance(valor, (list, tuple)):
        return [dados_portaveis(item) for item in valor]
    if hasattr(valor, "tolist"):
        return dados_portaveis(valor.tolist())
    return valor

def mostrar(nome, figura):
    figura.update_layout(font=dict(family="Arial, sans-serif", size=13), separators=",.")
    figura = go.Figure(dados_portaveis(figura.to_dict()))
    FIGURAS[nome] = figura
    figura.show(renderer="plotly_mimetype")

In [2]:
fontes = [
    ("População IBGE", PROJECT_ROOT / "sinan/processados/ibge_populacao_municipio_ano.parquet"),
    (f"SINAN {ANO - 1}", PROJECT_ROOT / f"sinan/DENGBR{(ANO - 1) % 100:02d}.parquet"),
    (f"SINAN {ANO}", PROJECT_ROOT / f"sinan/DENGBR{ANO % 100:02d}.parquet"),
    (f"SINAN {ANO + 1}", PROJECT_ROOT / f"sinan/DENGBR{(ANO + 1) % 100:02d}.parquet"),
    ("INMET (opcional para execução parcial)", CAMINHO_INMET),
]
display(pl.DataFrame({
    "fonte": [nome for nome, _ in fontes],
    "arquivo_disponivel": [caminho.exists() for _, caminho in fontes],
    "caminho": [str(caminho.relative_to(PROJECT_ROOT)) if caminho.is_relative_to(PROJECT_ROOT) else str(caminho) for _, caminho in fontes],
}))
if not CAMINHO_INMET.exists():
    display(Markdown("**INMET ausente:** serão gerados os agregados SINAN–IBGE. As variáveis climáticas permanecerão nulas até a fonte estar disponível e este notebook ser reexecutado."))

fonte,arquivo_disponivel,caminho
str,bool,str
"""População IBGE""",true,"""sinan/processados/ibge_populac…"
"""SINAN 2009""",true,"""sinan/DENGBR09.parquet"""
"""SINAN 2010""",true,"""sinan/DENGBR10.parquet"""
"""SINAN 2011""",true,"""sinan/DENGBR11.parquet"""
"""INMET (opcional para execução …",true,"""data/INMET_dataset_agregado.pa…"


## 2. Critérios da integração

**SINAN:** a unidade é a notificação por **município de residência**. As classificações
finais usam a harmonização do projeto, sem empregar antecedente de dengue como confirmação.
Todos os arquivos anuais disponíveis são examinados, e entram nas contagens apenas as
datas efetivas de notificação do ano escolhido. Assim, uma data de janeiro de 2010 em um
arquivo de 2009 entra em janeiro de 2010. São exigidos o ano-alvo e seus vizinhos; a lista
completa de arquivos consultados fica nos metadados. A série representa os registros
encontrados nessa coorte, sem garantir que todas as notificações existentes foram capturadas.
Não há deduplicação de pessoas.

**Exclusões explícitas:** residência ilegível ou fora do catálogo municipal do Censo
não é atribuída a uma UF. Datas ilegíveis não são inventadas. As exclusões relevantes ao
recorte são exportadas individualmente; as demais datas permanecem na auditoria por arquivo.
Zeros representam ausência de registros elegíveis na coorte presente, e não ausência de transmissão.

**IBGE:** a população estadual é a soma das populações municipais do mesmo Censo.
Os percentuais de saneamento são consultados **diretamente no nível estadual do SIDRA**,
preservando o universo de domicílios de cada tabela. O Censo é constante nos 12 meses;
essa repetição não constitui 12 novas medições independentes de saneamento.

**INMET:** reutiliza-se a triagem do projeto, que exige 24 intervalos horários válidos
para chuva diária e pelo menos 18 para temperatura/umidade. A precipitação é atribuída
ao início do intervalo; a leitura de 00:00 do primeiro dia do ano seguinte é incluída,
quando disponível. O painel fixo mantém as mesmas estações e pesos dentro de cada UF/ano.
Somam-se as médias diárias de chuva do painel somente em meses completos, equivalendo
à média dos acumulados mensais das estações. Valores ausentes permanecem nulos.
Dias meteorológicos são UTC; datas administrativas do SINAN não são convertidas em UTC.

In [3]:
print(f"Agregando as fontes para {ANO}...", flush=True)
resultados = build_aggregations(
    year=ANO,
    weather_path=CAMINHO_INMET,
    refresh_sidra=ATUALIZAR_SIDRA,
    fixed_stations=ESTACOES_FIXAS,
    minimum_station_coverage=COBERTURA_MINIMA_ESTACAO,
    minimum_monthly_coverage=COBERTURA_MINIMA_MENSAL,
)
mensal = resultados["mensal"]
anual = resultados["anual"]
auditoria = resultados["auditoria"]
excluidos = resultados["excluidos"]
metadados = resultados["metadados"]

display(pl.DataFrame({
    "indicador": ["Linhas mensais", "UFs", "Notificações elegíveis", "Confirmados elegíveis", "Linhas mensais com as três fontes", "Registros excluídos auditados"],
    "valor": [mensal.height, anual.height, int(mensal["notificacoes"].sum()), int(mensal["casos_confirmados"].sum()), int(mensal["tres_fontes_disponiveis"].sum()), excluidos.height],
}))
display(Markdown(f"**Situação INMET:** `{metadados['situacao_inmet']}`. **Esgotamento neste recorte:** {metadados['esgotamento']}."))

Agregando as fontes para 2010...


indicador,valor
str,i64
"""Linhas mensais""",324
"""UFs""",27
"""Notificações elegíveis""",1381060
"""Confirmados elegíveis""",869169
"""Linhas mensais com as três fon…",198
"""Registros excluídos auditados""",194


**Situação INMET:** `arquivo_processado`. **Esgotamento neste recorte:** rede ou fossa séptica.

## 3. Auditoria e reconciliação

A tabela abaixo distingue incluídos, datas de outros anos e exclusões. Registros de
outros anos-fonte incluídos no ano de calendário escolhido são contados separadamente.
Esta base pode diferir dos contratos anuais antigos, que usam o ano do arquivo.

In [4]:
display(auditoria.filter((pl.col("ano_arquivo") == ANO) | (pl.col("situacao") == "incluido")))
display(excluidos.group_by("situacao").len(name="registros").sort("situacao"))
display(anual.select("UF", "notificacoes_outro_ano_fonte").filter(pl.col("notificacoes_outro_ano_fonte") > 0))

assert mensal.height == 27 * 12
assert mensal.select(pl.struct("UF", "ano", "mes").n_unique()).item() == mensal.height
assert anual.height == 27
assert mensal["notificacoes"].sum() == auditoria.filter(pl.col("situacao") == "incluido")["notificacoes"].sum()
assert mensal["casos_confirmados"].sum() == anual["casos_confirmados"].sum()
assert mensal["populacao"].min() > 0
assert mensal.select(SANITATION).null_count().sum_horizontal().sum() == 0
print("Verificações de grade, denominadores, saneamento e reconciliação concluídas.")

arquivo,ano_arquivo,situacao,notificacoes,casos_confirmados
str,i32,str,u32,u32
"""DENGBR10.parquet""",2010,"""incluido""",1381060,869169
"""DENGBR10.parquet""",2010,"""residencia_invalida""",127,44
"""DENGBR10.parquet""",2010,"""residencia_sem_ibge""",67,35


situacao,registros
str,u32
"""residencia_invalida""",127
"""residencia_sem_ibge""",67


UF,notificacoes_outro_ano_fonte
str,u32


Verificações de grade, denominadores, saneamento e reconciliação concluídas.


## 4. Bases mensal e anual

Taxa mensal = confirmados do mês ÷ população residente do ano × 100.000.
Taxa anual = soma dos confirmados mensais ÷ a mesma população × 100.000.
A população e os percentuais censitários aparecem uma única vez no resumo anual.
Chuva anual exige os 12 meses completos; médias anuais exigem 12 médias mensais elegíveis
e são ponderadas pelo número de dias válidos. Não se extrapolam meses ausentes.

In [5]:
display(mensal.select(
    "UF", "ano", "mes", "casos_confirmados", "populacao", "taxa_confirmados_100k",
    *SANITATION, *WEATHER, "situacao_inmet", "tres_fontes_disponiveis",
).head(12))
display(anual.sort("taxa_confirmados_100k", descending=True))

UF,ano,mes,casos_confirmados,populacao,taxa_confirmados_100k,pct_agua_rede,pct_esgotamento,pct_coleta_lixo,temperatura_media_c,chuva_acumulada_mm,umidade_media_pct,situacao_inmet,tres_fontes_disponiveis
str,i32,i32,u32,u64,f64,f64,f64,f64,f64,f64,f64,str,bool
"""AC""",2010,1,3315,733559,451.906391,47.27,36.55,74.8,25.627081,null,86.428112,"""cobertura_insuficiente""",false
"""AC""",2010,2,7723,733559,1052.812385,47.27,36.55,74.8,25.929613,138.133333,86.71627,"""disponivel""",true
"""AC""",2010,3,6539,733559,891.407508,47.27,36.55,74.8,25.850806,129.4,87.373656,"""disponivel""",true
"""AC""",2010,4,3006,733559,409.78299,47.27,36.55,74.8,26.092639,75.066667,83.037963,"""disponivel""",true
"""AC""",2010,5,1536,733559,209.39011,47.27,36.55,74.8,24.666846,33.533333,83.675179,"""disponivel""",true
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""AC""",2010,8,457,733559,62.299011,47.27,36.55,74.8,25.393873,null,66.954992,"""cobertura_insuficiente""",false
"""AC""",2010,9,500,733559,68.160843,47.27,36.55,74.8,26.722083,37.266667,71.917593,"""disponivel""",true
"""AC""",2010,10,901,733559,122.82584,47.27,36.55,74.8,26.121505,75.0,77.270609,"""disponivel""",true


UF,ano,regiao,populacao,pct_agua_rede,pct_esgotamento,pct_coleta_lixo,notificacoes,casos_confirmados,notificacoes_outro_ano_fonte,meses_clima_completo,chuva_anual_mm,temperatura_media_c,umidade_media_pct,taxa_confirmados_100k,taxa_notificacoes_100k
str,i32,str,u64,f64,f64,f64,u32,u32,u32,u32,f64,f64,f64,f64,f64
"""AC""",2010,"""Norte""",733559,47.27,36.55,74.8,42124,34052,0,8,null,null,null,4642.026067,5742.414721
"""MS""",2010,"""Centro-Oeste""",2449024,82.87,38.74,86.45,86729,56521,0,7,null,22.863459,69.116233,2307.898983,3541.36995
"""RR""",2010,"""Norte""",450479,81.23,44.59,78.85,11785,6990,0,0,null,null,null,1551.681654,2616.104191
"""GO""",2010,"""Centro-Oeste""",6003788,79.31,48.91,91.02,117924,74556,0,10,null,23.915554,63.962816,1241.816,1964.15996
"""AL""",2010,"""Nordeste""",3120494,68.62,32.64,79.76,54488,37877,0,9,null,25.268681,78.760346,1213.814223,1746.133785
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""RN""",2010,"""Nordeste""",3168027,86.38,45.22,84.48,9422,3150,0,7,null,null,null,99.430971,297.409081
"""MA""",2010,"""Nordeste""",6574789,65.88,26.66,55.83,6857,4810,0,8,null,27.196276,70.14011,73.158241,104.29232
"""RS""",2010,"""Sul""",10693929,85.33,74.57,92.08,4944,3493,0,10,null,19.71129,75.458707,32.663392,46.231839


## 5. Sazonalidade das taxas por UF

Cada célula representa a taxa de confirmados de uma UF em um mês. A ordenação segue
a taxa anual; os valores aparecem ao passar o cursor. O gráfico permite localizar picos
e comparar a distribuição mensal sem confundir estados grandes com taxas elevadas.

In [6]:
ordem_ufs = anual.sort("taxa_confirmados_100k", descending=True)["UF"].to_list()
matriz_taxas = mensal.to_pandas().pivot(index="UF", columns="mes", values="taxa_confirmados_100k").reindex(index=ordem_ufs, columns=range(1, 13))
fig = px.imshow(
    matriz_taxas.to_numpy(), x=MESES, y=ordem_ufs,
    color_continuous_scale="YlOrRd", aspect="auto",
    labels={"x": "Mês de notificação", "y": "UF", "color": "Confirmados/100 mil"},
    title=f"Dengue: taxas mensais por UF — {ANO}",
)
fig.update_traces(hovertemplate="UF: %{y}<br>Mês: %{x}<br>Confirmados/100 mil: %{z:.2f}<extra></extra>")
fig.update_layout(height=760)
mostrar("01_taxas_mensais", fig)

## 6. Painel temporal e contexto de saneamento

Altere `UF_DESTAQUE` na configuração e reexecute as células seguintes para explorar outro estado.
Os gráficos compartilham o calendário e preservam uma escala própria para cada variável.
Os indicadores censitários no título fornecem o contexto estadual; sua presença não
demonstra uma relação causal com os picos mensais.

In [7]:
if UF_DESTAQUE not in anual["UF"].to_list():
    raise ValueError("UF_DESTAQUE inválida: informe uma das 27 siglas de UF.")
estado = mensal.filter(pl.col("UF") == UF_DESTAQUE).sort("mes")
contexto = anual.filter(pl.col("UF") == UF_DESTAQUE).row(0, named=True)
resumo_saneamento = " · ".join(f"{ROTULOS_SANEAMENTO[c]}: {contexto[c]:.1f}%" for c in SANITATION)
display(Markdown(f"**{UF_DESTAQUE} — Censo {ANO}:** {resumo_saneamento}"))

fig = make_subplots(rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.07,
                    subplot_titles=["Dengue — confirmados por 100 mil", "Chuva acumulada do painel fixo", "Temperatura média", "Umidade relativa média"])
fig.add_trace(go.Scatter(x=MESES, y=estado["taxa_confirmados_100k"].to_list(), mode="lines+markers", line_color="#2878b5", name="Dengue"), row=1, col=1)
for linha, coluna, unidade, cor in [
    (2, "chuva_acumulada_mm", "mm", "#228b69"),
    (3, "temperatura_media_c", "°C", "#c46c2b"),
    (4, "umidade_media_pct", "%", "#8055a5"),
]:
    valores = estado[coluna].to_list()
    if any(v is not None for v in valores):
        trace = go.Bar(x=MESES, y=valores, marker_color=cor) if linha == 2 else go.Scatter(x=MESES, y=valores, mode="lines+markers", line_color=cor, connectgaps=False)
        fig.add_trace(trace, row=linha, col=1)
    else:
        fig.add_annotation(x=0.5, y=0.5, xref=f"x{linha} domain", yref=f"y{linha} domain",
                           text="Sem dados climáticos elegíveis neste recorte", showarrow=False, font_color="#777777")
    fig.update_yaxes(title_text=unidade, row=linha, col=1)
fig.update_yaxes(title_text="por 100 mil", row=1, col=1)
fig.update_layout(title=f"{UF_DESTAQUE}: dengue e clima ao longo de {ANO}", height=880, showlegend=False)
mostrar("02_painel_temporal", fig)

**SP — Censo 2010:** Água — rede geral: 95.0% · Esgoto — rede ou fossa séptica: 91.4% · Lixo — coleta direta ou em caçamba: 98.2%

## 7. Dispersão anual integrando as três fontes

Cada ponto é **uma UF no ano**. O eixo horizontal mostra saneamento, o vertical a taxa
anual de dengue, e a cor a temperatura média anual do painel fixo. A mesma amostra de
UFs completas é usada nos três painéis. Passe o cursor para consultar também chuva e umidade.

Quando não há UFs com clima anual elegível, apresenta-se explicitamente a comparação
**SINAN–IBGE**, sem atribuir valores climáticos aos pontos. Não se estimam efeitos causais
nem se tratam os meses repetidos como observações independentes de saneamento.

In [8]:
colunas_completas = [*SANITATION, "taxa_confirmados_100k", "chuva_anual_mm", "temperatura_media_c", "umidade_media_pct"]
amostra_tres_fontes = anual.drop_nulls(colunas_completas)
usar_clima = amostra_tres_fontes.height > 0
amostra = amostra_tres_fontes if usar_clima else anual.drop_nulls([*SANITATION, "taxa_confirmados_100k"])
descricao = "SINAN + IBGE + INMET" if usar_clima else "SINAN + IBGE — clima anual indisponível"
display(Markdown(f"**Amostra:** {amostra.height} UFs. **Fontes representadas:** {descricao}."))

longa = amostra.to_pandas().melt(
    id_vars=["UF", "regiao", "populacao", "taxa_confirmados_100k", "temperatura_media_c", "chuva_anual_mm", "umidade_media_pct"],
    value_vars=SANITATION, var_name="indicador", value_name="cobertura_domiciliar_pct",
)
longa["indicador"] = longa["indicador"].map(ROTULOS_SANEAMENTO)
opcoes_cor = dict(color="temperatura_media_c", color_continuous_scale="YlOrRd") if usar_clima else dict(color="regiao", color_discrete_map=CORES_REGIAO)
fig = px.scatter(
    longa, x="cobertura_domiciliar_pct", y="taxa_confirmados_100k", facet_col="indicador",
    hover_name="UF", hover_data={"populacao": ":,", "chuva_anual_mm": ":.1f", "umidade_media_pct": ":.1f"},
    labels={"cobertura_domiciliar_pct": "% de domicílios", "taxa_confirmados_100k": "Confirmados/100 mil", "temperatura_media_c": "Temperatura (°C)", "regiao": "Região"},
    title=f"Dengue e saneamento por UF — {ANO}<br><sup>{descricao}</sup>",
    **opcoes_cor,
)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_annotations(font_size=11)
fig.update_traces(marker_size=10, marker_line_width=0.7, marker_line_color="white")
fig.update_xaxes(range=[0, 102])
fig.update_yaxes(range=[0, max(1.0, float(amostra["taxa_confirmados_100k"].max()) * 1.1)], autorange=False)
fig.update_layout(height=560, autosize=True, margin=dict(t=120, b=115, l=85, r=30), legend=dict(orientation="h", y=-0.24))
mostrar("03_dispersao_integrada", fig)

**Amostra:** 27 UFs. **Fontes representadas:** SINAN + IBGE — clima anual indisponível.

## 7.1. Comparação mensal com as três fontes

A chuva anual exige 12 meses completos; lacunas horárias podem impedir essa comparação mesmo quando vários meses têm dados válidos. Este gráfico usa **uma observação por UF no mês escolhido**, mantendo os critérios de cobertura.

Altere `MES_DISPERSAO` na configuração. Nos três painéis, o eixo horizontal mostra o indicador censitário de saneamento, o vertical a taxa mensal de confirmados e a cor a temperatura média do mês. Chuva, umidade e população aparecem ao passar o cursor. A amostra é comum aos painéis e exige as três variáveis climáticas válidas. O mês padrão é março; ele não foi escolhido por maximizar correlações. A comparação é exploratória e não estima efeito causal.

In [9]:
if not isinstance(MES_DISPERSAO, int) or not 1 <= MES_DISPERSAO <= 12:
    raise ValueError("MES_DISPERSAO deve estar entre 1 e 12.")
amostra_mensal = mensal.filter(
    (pl.col("mes") == MES_DISPERSAO) & pl.col("tres_fontes_disponiveis")
)
display(Markdown(f"**{MESES[MES_DISPERSAO - 1]}/{ANO}: {amostra_mensal.height} UFs com as três fontes.** Cada ponto representa uma UF neste mês."))
if amostra_mensal.height:
    longa_mensal = amostra_mensal.to_pandas().melt(
        id_vars=["UF", "regiao", "populacao", "taxa_confirmados_100k", "temperatura_media_c", "chuva_acumulada_mm", "umidade_media_pct"],
        value_vars=SANITATION, var_name="indicador", value_name="cobertura_domiciliar_pct",
    )
    longa_mensal["indicador"] = longa_mensal["indicador"].map(ROTULOS_SANEAMENTO)
    fig = px.scatter(
        longa_mensal, x="cobertura_domiciliar_pct", y="taxa_confirmados_100k",
        color="temperatura_media_c", color_continuous_scale="YlOrRd", facet_col="indicador",
        hover_name="UF", hover_data={"populacao": ":,", "chuva_acumulada_mm": ":.1f", "umidade_media_pct": ":.1f"},
        labels={"cobertura_domiciliar_pct": "% de domicílios", "taxa_confirmados_100k": "Confirmados/100 mil", "temperatura_media_c": "Temp. (°C)"},
        title=f"Dengue, saneamento e temperatura — {MESES[MES_DISPERSAO - 1]}/{ANO}<br><sup>{amostra_mensal.height} UFs · amostra comum com SINAN, IBGE e INMET</sup>",
    )
    fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1], font_size=10))
    fig.update_traces(marker_size=10, marker_line_width=0.7, marker_line_color="white")
    fig.update_xaxes(range=[0, 102])
    fig.update_yaxes(range=[0, max(1.0, float(amostra_mensal["taxa_confirmados_100k"].max()) * 1.1)], autorange=False)
    fig.update_layout(height=560, autosize=True, margin=dict(t=120, b=100, l=85, r=35))
    mostrar("05_dispersao_mensal_tres_fontes", fig)
else:
    display(Markdown("Não há UFs com as três variáveis climáticas elegíveis neste mês. Consulte a cobertura meteorológica abaixo."))

**Mar/2010: 16 UFs com as três fontes.** Cada ponto representa uma UF neste mês.

## 8. Cobertura meteorológica e elegibilidade

Cada mapa de calor informa a porcentagem de dias com dados válidos do **painel fixo**,
independentemente de a média mensal ter atingido o limiar de publicação. Zero significa
**nenhum dia elegível**, inclusive quando o arquivo INMET está ausente; não significa
temperatura, umidade ou precipitação igual a zero. A tabela distingue arquivo ausente,
ausência de painel e cobertura insuficiente. Ter estações elegíveis não garante que o
painel represente toda a área ou a população estadual.

In [10]:
display(mensal.group_by("situacao_inmet").len(name="uf_meses").sort("situacao_inmet"))
display(mensal.group_by("UF").agg(
    pl.col("n_estacoes_painel_fixo").max(), pl.col("estacoes_painel_fixo").drop_nulls().first(),
    pl.col("tres_fontes_disponiveis").sum().alias("meses_com_tres_fontes"),
).sort("UF"))

fig = make_subplots(rows=1, cols=3, shared_yaxes=True,
                    subplot_titles=["Temperatura", "Chuva", "Umidade"], horizontal_spacing=0.06)
for coluna, nome in enumerate(WEATHER, start=1):
    matriz = mensal.to_pandas().pivot(index="UF", columns="mes", values=f"cobertura_{nome}_pct").reindex(index=ordem_ufs, columns=range(1, 13))
    fig.add_trace(go.Heatmap(
        x=MESES, y=ordem_ufs, z=matriz.to_numpy(), coloraxis="coloraxis",
        hovertemplate="UF: %{y}<br>Mês: %{x}<br>Dias válidos: %{z:.1f}%<extra></extra>",
    ), row=1, col=coluna)
fig.update_yaxes(autorange="reversed")
fig.update_layout(
    title=f"Cobertura diária do painel meteorológico fixo — {ANO}<br><sup>Situação do arquivo: {metadados['situacao_inmet']}</sup>",
    height=760, autosize=True,
    coloraxis=dict(cmin=0, cmax=100, colorscale=[[0, "#ededed"], [0.5, "#8bc4ac"], [1, "#146c50"]], colorbar_title="Dias válidos (%)"),
)
mostrar("04_cobertura_inmet", fig)

situacao_inmet,uf_meses
str,u32
"""cobertura_insuficiente""",78
"""disponivel""",198
"""sem_painel_elegivel""",48


UF,n_estacoes_painel_fixo,estacoes_painel_fixo,meses_com_tres_fontes
str,u32,str,u32
"""AC""",3,"""A104,A136,A138""",8
"""AL""",3,"""A303,A327,A356""",9
"""AM""",3,"""A122,A126,A128""",10
"""AP""",0,null,0
"""BA""",3,"""A416,A431,A433""",8
…,…,…,…
"""RS""",3,"""A808,A832,A852""",10
"""SC""",3,"""A814,A816,A859""",8
"""SE""",3,"""A417,A419,A420""",10


## 9. Exportação

Os resultados ficam em `sinan/processados/agregacao_tres_fontes/<ano>/`, diretório
de dados ignorado pelo Git. Os gráficos HTML são interativos e compartilham uma cópia
local do Plotly na pasta `figuras/`; mantenha a pasta completa ao compartilhar.
O notebook conserva os gráficos e tabelas da última execução.

In [11]:
export_aggregations(resultados, DESTINO)
pasta_figuras = DESTINO / "figuras"
pasta_figuras.mkdir(parents=True, exist_ok=True)
for nome, figura in FIGURAS.items():
    figura.write_html(pasta_figuras / f"{nome}.html", include_plotlyjs="directory")
display(pl.DataFrame({"arquivo": [str(p.relative_to(PROJECT_ROOT)) for p in sorted(DESTINO.rglob("*")) if p.is_file() and p.name != "plotly.min.js"]}))

arquivo
str
"""sinan/processados/agregacao_tr…"
"""sinan/processados/agregacao_tr…"
"""sinan/processados/agregacao_tr…"
"""sinan/processados/agregacao_tr…"
"""sinan/processados/agregacao_tr…"
…
"""sinan/processados/agregacao_tr…"
"""sinan/processados/agregacao_tr…"
"""sinan/processados/agregacao_tr…"


## Limites de interpretação

São associações ecológicas e descritivas: notificações dependem de transmissão,
diagnóstico, acesso à saúde e vigilância. Sazonalidade, autocorrelação e atrasos de
notificação podem produzir associações climáticas. Não se compara a magnitude de
correlações de escalas diferentes nem se conclui qual fonte "explica mais" dengue.
Os gráficos temporais usam o mesmo mês; uma investigação de defasagens deve definir
as janelas antecipadamente e considerar a dependência temporal.

Em 2010, esgotamento reúne rede ou fossa séptica; em 2022, reúne rede geral/pluvial
ou fossa ligada à rede. Água também tem definições diferentes entre os Censos.
Por isso, reexecutar para 2022 gera outro recorte exploratório, sem constituir uma
comparação harmonizada da evolução do saneamento.

Fontes e lógica: [SINAN/DATASUS](https://datasus.saude.gov.br/transferencia-de-arquivos/),
[SIDRA 3218](https://sidra.ibge.gov.br/tabela/3218),
[SIDRA 6803](https://sidra.ibge.gov.br/tabela/6803),
[SIDRA 6805](https://sidra.ibge.gov.br/tabela/6805),
[SIDRA 6892](https://sidra.ibge.gov.br/tabela/6892),
[INMET](https://portal.inmet.gov.br/dadoshistoricos) e `utils/aggregate_sources.py`.
As URLs exatas das consultas estaduais e a lista dos arquivos SINAN ficam em `metadados.json`.